<a href="https://colab.research.google.com/github/jeansdelgado/pagesegmentacion/blob/main/tarea4jjdr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installing RAG Dependencies

Este bloque de código se encarga de instalar las dependencias necesarias para construir una aplicación de Generación Aumentada por Recuperación (RAG). Utiliza `pip` para instalar varias bibliotecas:

*   `transformers` y `datasets`: Son de Hugging Face y se usan para modelos de procesamiento de lenguaje natural y manejo de conjuntos de datos.
*   `sentence-transformers`: Para crear embeddings (representaciones numéricas) de textos.
*   `faiss-cpu`: Una biblioteca para realizar búsquedas eficientes de similitud en vectores, esencial para la base de datos vectorial de RAG.
*   `langchain-google-genai` y `langchain-community`: Son componentes de LangChain que facilitan la integración con modelos de IA de Google (como Gemini) y proporcionan utilidades para construir aplicaciones basadas en LLM.

También incluye comentarios sobre la instalación opcional de `accelerate` y `bitsandbytes`, que son útiles para optimizar el rendimiento y el uso de memoria con modelos más grandes. Al final, imprime un mensaje de confirmación cuando todas las dependencias se han instalado.


In [3]:
import sys
!{sys.executable} -m pip install -q transformers datasets sentence-transformers faiss-cpu langchain-google-genai langchain-community gradio

# Optional: Install accelerate and bitsandbytes for faster inference and quantization with large models
# !{sys.executable} -m pip install -q accelerate bitsandbytes

print("✅ Dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ Dependencias instaladas


### Descarga y Preparación de Documentos

Vamos a descargar el contenido HTML de las URLs proporcionadas y asignaremos los segmentos correspondientes a cada una.

In [ ]:
import requests
from bs4 import BeautifulSoup

# Define the pages and their corresponding segments
pages_to_download = [
    {
        "url": "https://www.digitel.com.ve/",
        "segments": ["Personas", "Emprendedores"]
    },
    {
        "url": "https://www.digitel.com.ve/pymes",
        "segments": ["Pymes", "Negocios", "Pyme Social"]
    },
    {
        "url": "https://www.digitel.com.ve/empresas",
        "segments": ["Empresas", "Gobiernos", "Carrier"]
    }
]

# List to store the downloaded documents
documentos = [] # Cambiado 'documents' a 'documentos' para coincidir con la solicitud de 'bloque de código [1]'

print("Iniciando descarga de documentos...")

try:
    for page_info in pages_to_download:
        url = page_info["url"]
        segments = page_info["segments"]

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            html_content = response.text

            # Optional: You might want to parse the HTML to get only visible text
            # soup = BeautifulSoup(html_content, 'html.parser')
            # text_content = soup.get_text(separator=' ', strip=True)

            documentos.append({
                "url": url,
                "segments": segments,
                "html_content": html_content,
                # "text_content": text_content # if you decide to extract text
            })

            print(f"✅ Descargado {url} (Longitud HTML: {len(html_content)} caracteres) - Segmentos: {', '.join(segments)}")

        except requests.exceptions.RequestException as e:
            print(f"❌ Error al descargar {url}: {e}")

except Exception as e:
    print(f"❌ Un error inesperado ocurrió durante el procesamiento: {e}")
finally:
    print("\n--- Descarga de documentos completada ---")
    # Print the trace (summary of downloaded documents)
    print("\n--- Traza de Documentos Descargados ---")
    for doc in documentos:
        print(f"URL: {doc['url']}")
        print(f"  Segmentos: {', '.join(doc['segments'])}")
        print(f"  Longitud HTML: {len(doc['html_content'])} caracteres\n")

    print(f"Total de documentos procesados: {len(documentos)}")

Iniciando descarga de documentos...
✅ Descargado https://www.digitel.com.ve/ (Longitud HTML: 101531 caracteres) - Segmentos: Personas, Emprendedores
✅ Descargado https://www.digitel.com.ve/pymes (Longitud HTML: 54760 caracteres) - Segmentos: Pymes, Negocios, Pyme Social
✅ Descargado https://www.digitel.com.ve/empresas (Longitud HTML: 47938 caracteres) - Segmentos: Empresas, Gobiernos, Carrier

--- Descarga de documentos completada ---

--- Traza de Documentos Descargados ---
URL: https://www.digitel.com.ve/
  Segmentos: Personas, Emprendedores
  Longitud HTML: 101531 caracteres

URL: https://www.digitel.com.ve/pymes
  Segmentos: Pymes, Negocios, Pyme Social
  Longitud HTML: 54760 caracteres

URL: https://www.digitel.com.ve/empresas
  Segmentos: Empresas, Gobiernos, Carrier
  Longitud HTML: 47938 caracteres

Total de documentos procesados: 3


### Configuración de Embeddings y Creación de Base de Datos Vectorial

Aquí configuraremos el modelo de embeddings de Google Gemini y crearemos una base de datos vectorial FAISS con los documentos procesados.

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from bs4 import BeautifulSoup

# Obtener la clave API de los secretos de Colab
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

# Configurar la API de Gemini
genai.configure(api_key=GOOGLE_API_KEY)

print("✅ API de Google Gemini configurada.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ API de Google Gemini configurada.


In [ ]:
# Inicializar el modelo de embeddings
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY)

print("✅ Modelo de embeddings inicializado: models/gemini-embedding-001")

✅ Modelo de embeddings inicializado: models/gemini-embedding-001


#### Procesamiento de Documentos para Embeddings

Antes de crear los embeddings, necesitamos extraer el texto plano de los documentos HTML y dividirlos en fragmentos más pequeños para un procesamiento eficiente.

In [ ]:
processed_docs = []
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Tamaño de cada fragmento de texto
    chunk_overlap=200 # Superposición entre fragmentos para mantener el contexto
)

print("Extrayendo texto y creando objetos Document...")
try:
    for doc_info in documentos: # Changed 'documents' to 'documentos' to match variable name
        html_content = doc_info['html_content']
        url = doc_info['url']
        segments = doc_info['segments']

        soup = BeautifulSoup(html_content, 'html.parser')
        text_content = soup.get_text(separator=' ', strip=True)

        # Crear un objeto Document de Langchain con los metadatos
        langchain_doc = Document(
            page_content=text_content,
            metadata={
                "source": url,
                "segments": segments
            }
        )
        processed_docs.append(langchain_doc)
        print(f"✅ Procesando documento desde: {url} con segmentos: {', '.join(segments)}")

    # Dividir los documentos en chunks
    chunks = text_splitter.split_documents(processed_docs)

    print(f"✅ Documentos procesados. Total de chunks creados: {len(chunks)}")
    if chunks:
        print("Primer chunk de ejemplo:")
        print(chunks[0].page_content[:200] + "...")
        print(f"Metadatos del primer chunk: {chunks[0].metadata}")
    else:
        print("No se crearon chunks.")

except Exception as e:
    print(f"❌ Un error ocurrió durante el procesamiento de documentos: {e}")
finally:
    print("--- Proceso de extracción y chunking finalizado ---")

Extrayendo texto y creando objetos Document...
✅ Procesando documento desde: https://www.digitel.com.ve/ con segmentos: Personas, Emprendedores
✅ Procesando documento desde: https://www.digitel.com.ve/pymes con segmentos: Pymes, Negocios, Pyme Social
✅ Procesando documento desde: https://www.digitel.com.ve/empresas con segmentos: Empresas, Gobiernos, Carrier
✅ Documentos procesados. Total de chunks creados: 39
Primer chunk de ejemplo:
Digitel | Conectividad sin límites Personas Pymes Empresas Recargar  Pagar factura  App Digitel Tu Cuenta Internet 5G Planes y paquetes LDI Roaming de Datos eSIM VoLTE  Contáctanos Personas Pymes E...
Metadatos del primer chunk: {'source': 'https://www.digitel.com.ve/', 'segments': ['Personas', 'Emprendedores']}
--- Proceso de extracción y chunking finalizado ---


#### Creación de la Base de Datos Vectorial FAISS

Ahora, utilizaremos los chunks de texto y el modelo de embeddings para crear la base de datos vectorial FAISS. Esta base de datos nos permitirá buscar fragmentos de texto relevantes de manera eficiente.

In [ ]:
# Crear la base de datos vectorial FAISS
vector_db = FAISS.from_documents(chunks, embeddings_model)

print("✅ Base de datos vectorial FAISS creada con éxito.")
print(f"Número de vectores en la base de datos: {vector_db.index.ntotal}")

✅ Base de datos vectorial FAISS creada con éxito.
Número de vectores en la base de datos: 39


### Añadir Nuevos Documentos a la Base de Datos Vectorial Existente

Si deseas actualizar tu base de datos vectorial FAISS con nuevos documentos sin reconstruirla desde cero, puedes seguir estos pasos:

1.  **Definir las nuevas URLs y segmentos** (si aplica).
2.  **Descargar y extraer el contenido** HTML/texto de estas URLs.
3.  **Dividir el nuevo contenido en chunks**.
4.  **Generar embeddings** para estos nuevos chunks utilizando el mismo modelo de embeddings.
5.  **Añadir los nuevos chunks (y sus embeddings) a la base de datos** `vector_db` existente.

A continuación, se muestra el código para realizar esta operación.

In [ ]:
# 1. Definir nuevas URLs y segmentos (ejemplo)
new_pages_to_download = [
    {
        "url": "https://www.digitel.com.ve/conocenos/nuestra-empresa",
        "segments": ["Corporativo", "Historia"]
    },
    {
        "url": "https://www.digitel.com.ve/telefonia/prepago",
        "segments": ["Personas", "Prepago"]
    }
]

# Lista para almacenar los nuevos documentos descargados
new_documentos = []

print("Iniciando descarga de NUEVOS documentos...")

try:
    for page_info in new_pages_to_download:
        url = page_info["url"]
        segments = page_info["segments"]

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            html_content = response.text

            new_documentos.append({
                "url": url,
                "segments": segments,
                "html_content": html_content,
            })

            print(f"✅ Descargado {url} (Longitud HTML: {len(html_content)} caracteres) - Segmentos: {', '.join(segments)}")

        except requests.exceptions.RequestException as e:
            print(f"❌ Error al descargar {url}: {e}")

except Exception as e:
    print(f"❌ Un error inesperado ocurrió durante el procesamiento de nuevos documentos: {e}")
finally:
    print("\n--- Descarga de nuevos documentos completada ---")
    print(f"Total de nuevos documentos procesados: {len(new_documentos)}")

#### Procesamiento de Nuevos Documentos para Embeddings

Similar a como se hizo con los documentos originales, extraemos el texto plano de los nuevos documentos HTML y los dividimos en fragmentos.

In [ ]:
# 3. y 4. Extraer texto, crear objetos Document y dividir en chunks para los nuevos documentos
new_processed_docs = []

print("Extrayendo texto y creando objetos Document para nuevos documentos...")
try:
    for doc_info in new_documentos:
        html_content = doc_info['html_content']
        url = doc_info['url']
        segments = doc_info['segments']

        soup = BeautifulSoup(html_content, 'html.parser')
        text_content = soup.get_text(separator=' ', strip=True)

        # Crear un objeto Document de Langchain con los metadatos
        langchain_doc = Document(
            page_content=text_content,
            metadata={
                "source": url,
                "segments": segments
            }
        )
        new_processed_docs.append(langchain_doc)
        print(f"✅ Procesando nuevo documento desde: {url} con segmentos: {', '.join(segments)}")

    # Dividir los nuevos documentos en chunks
    new_chunks = text_splitter.split_documents(new_processed_docs)

    print(f"✅ Nuevos documentos procesados. Total de nuevos chunks creados: {len(new_chunks)}")
    if new_chunks:
        print("Primer nuevo chunk de ejemplo:")
        print(new_chunks[0].page_content[:200] + "...")
        print(f"Metadatos del primer nuevo chunk: {new_chunks[0].metadata}")
    else:
        print("No se crearon nuevos chunks.")

except Exception as e:
    print(f"❌ Un error ocurrió durante el procesamiento de nuevos documentos: {e}")
finally:
    print("--- Proceso de extracción y chunking de nuevos documentos finalizado ---")

#### Actualización de la Base de Datos Vectorial FAISS

Finalmente, añadimos los embeddings de los nuevos chunks a la base de datos `vector_db` existente.

In [ ]:
# 5. Añadir los nuevos chunks a la base de datos FAISS existente
if new_chunks:
    vector_db.add_documents(new_chunks)
    print("✅ Nuevos documentos añadidos a la base de datos vectorial FAISS con éxito.")
    print(f"Número total de vectores en la base de datos ahora: {vector_db.index.ntotal}")
else:
    print("No hay nuevos chunks para añadir a la base de datos FAISS.")

In [29]:
import gradio as gr
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import re # For URL validation

# New import for zero-shot classification
from transformers import pipeline

# --- Zero-Shot Classifier Initialization (Global) ---
# Initialize the zero-shot classification pipeline once
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Candidate labels for Zero-Shot Classification
segments_options_list = ["Personas", "Emprendedores", "Pymes", "Negocios", "Pyme Social", "Empresas", "Gobiernos", "Prescriptor", "Revendedores","Especificador","Alidos","Proveedores", "Impulsadores","Empleados"]

# Default pages data for the Gradio Dataframe (modified to remove segments column)
default_pages_data_for_df_for_ui = [
    ["https://www.digitel.com.ve/"],
    ["https://www.digitel.com.ve/pymes"],
    ["https://www.digitel.com.ve/empresas"]
]

# Helper function to validate URLs
def is_valid_url(url):
    regex = re.compile(
        r'^(?:http|ftp)s?://' # http:// or https://
        r'(?:(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+(?:[A-Z]{2,6}\.?|[A-Z0-9-]{2,}\.?)|' # domain...
        r'localhost|' # localhost...
        r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})' # ...or ip
        r'(?::\d+)?' # optional port
        r'(?:/?|[/?]\S+\.(?:html?|asp|php|jsp|pdf|doc|docx|ppt|pptx|xls|xlsx|txt|xml|json|csv|rtf|zip|rar|jpg|jpeg|png|gif|bmp|tiff|svg|ico)(?:[?#]\S*)?|/?)$', re.IGNORECASE) # Allow various file extensions
    return re.match(regex, url) is not None

def rag_chat(api_key, temperature, selected_model, pages_data_list, chat_history, user_question, progress=gr.Progress()):
    """
    Backend function for the Gradio chatbot. It re-initializes the RAG pipeline
    based on the configuration provided in the Gradio UI.
    """
    progress(0, desc="Iniciando...")

    if not api_key:
        return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': "⚠️ Por favor, introduce tu API Key de Gemini en la barra lateral."}] , "⚠️ Por favor, introduce tu API Key de Gemini en la barra lateral.", ""

    # Configure Gemini API
    try:
        genai.configure(api_key=api_key)
    except Exception as e:
        return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': f"❌ Error al configurar la API de Gemini: {e}"}], f"❌ Error al configurar la API de Gemini: {e}", ""

    # Process pages_data_list from Gradio Dataframe (now only contains URLs)
    processed_pages = []
    for row in pages_data_list:
        url_link = row[0] # Only URL now

        if not url_link or not is_valid_url(url_link):
            return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': f"❌ URL inválida o vacía en la fila: {row}. Revísala en 'Definir páginas a descargar'."}], f"❌ URL inválida o vacía en la fila: {row}. Revísala en 'Definir páginas a descargar'.", ""

        processed_pages.append({"url": url_link})

    if not processed_pages:
        return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': "⚠️ No hay páginas definidas para descargar. Por favor, añade URLs en la barra lateral."}] , "⚠️ No hay páginas definidas para descargar. Por favor, añade URLs en la barra lateral.", ""

    documentos = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    progress(0.1, desc="Descargando y procesando documentos...")
    # --- Document Download and Processing ---
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9"
    }
    for i, page_info in enumerate(processed_pages):
        url = page_info["url"]
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            html_content = response.text

            soup = BeautifulSoup(html_content, 'html.parser')
            text_content = soup.get_text(separator=' ', strip=True)

            documentos.append({
                "url": url,
                "html_content": html_content,
                "text_content": text_content
            })
            progress(0.1 + 0.2 * (i + 1) / len(processed_pages), desc=f"Procesando {url}...")
        except requests.exceptions.ConnectionError as e:
            print(f"❌ Error de conexión al descargar {url}: {e}")
        except requests.exceptions.Timeout as e:
            print(f"❌ Tiempo de espera agotado al descargar {url}: {e}")
        except requests.exceptions.HTTPError as e:
            print(f"❌ Error HTTP al descargar {url}: {e}")
        except requests.exceptions.RequestException as e:
            print(f"❌ Error de solicitud al descargar {url}: {e}")
        except Exception as e:
            print(f"❌ Un error inesperado ocurrió al procesar la URL {url}: {e}")

    if not documentos:
        return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': "⚠️ No se pudo descargar ni procesar ningún documento. Por favor, revisa las URLs."}] , "⚠️ No se pudo descargar ni procesar ningún documento. Por favor, revisa las URLs.", ""

    processed_docs = []
    for doc_info in documentos:
        langchain_doc = Document(
            page_content=doc_info['text_content'],
            metadata={
                "source": doc_info['url'] # No segments in metadata now
            }
        )
        processed_docs.append(langchain_doc)

    progress(0.3, desc="Dividiendo documentos en chunks...")
    chunks = text_splitter.split_documents(processed_docs)

    if not chunks:
        return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': "⚠️ No se pudieron generar chunks a partir de los documentos descargados. "
                                               "Asegúrate de que las URLs contienen contenido textual significativo."
                                               "No se encontraron documentos relevantes para su pregunta. Intente con otra." } ], \
               "⚠️ No se pudieron generar chunks a partir de los documentos descargados. " \
               "Asegúrate de que las URLs contienen contenido textual significativo.", ""

    progress(0.4, desc="Generando embeddings...")
    # Initialize embeddings model
    embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=api_key)

    progress(0.6, desc="Creando base de datos vectorial FAISS...")
    # Create FAISS vector DB
    vector_db = FAISS.from_documents(chunks, embeddings_model)

    progress(0.8, desc="Configurando LLM y cadena RAG...")
    # Configure LLM
    llm = ChatGoogleGenerativeAI(
        model=selected_model,
        google_api_key=api_key,
        temperature=temperature,
        output_version="v1"
    )

    # Define the RAG prompt
    template = """Responde la pregunta basándose únicamente en el siguiente contexto:
    {context}

    Pregunta: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)

    # Create the RAG chain
    rag_chain = (
        {"context": vector_db.as_retriever(search_kwargs={"k": 3}), "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Invoke the RAG chain
    try:
        progress(0.9, desc="Obteniendo respuesta del LLM...")
        response_text = rag_chain.invoke(user_question)
    except Exception as e:
        return chat_history + [{'role': 'user', 'content': user_question}, {'role': 'assistant', 'content': f"❌ Error al obtener respuesta del LLM: {e}"}], f"❌ Error al obtener respuesta del LLM: {e}", ""

    # Heuristic to determine if the LLM's response is an unsatisfactory answer
    unhelpful_phrases = [
        "no puedo encontrar información",
        "no está disponible en el contexto",
        "no se proporciona esa información",
        "no tengo información sobre",
        "no sé",
        "lo siento",
        "basándome únicamente en el contexto proporcionado, no puedo responder"
    ]

    is_unhelpful = False
    if not response_text.strip(): # Check if response is empty
        is_unhelpful = True
    elif len(response_text.strip()) < 50: # Check if response is too short and might be generic
        is_unhelpful = True
    else:
        for phrase in unhelpful_phrases:
            if phrase in response_text.lower():
                is_unhelpful = True
                break

    if is_unhelpful:
        response_text = (
            "Lo siento, no pude encontrar una respuesta específica a tu pregunta en la información proporcionada "
            "en las páginas de la empresa que tengo acceso. "
            "Mi conocimiento se basa exclusivamente en el contenido de esas páginas web. "
            "Te sugiero que explores directamente la página web o contactes con los canales de comunicación de la empresa "
            "para obtener ayuda más detallada."
        )

    progress(0.95, desc="Clasificando segmentos...")
    # --- Zero-Shot Classification for Segments ---
    classification_results = classifier(user_question, candidate_labels=segments_options_list)

    classified_segments_with_scores = []
    # Sort by score in descending order to pick the most relevant ones first
    sorted_labels_scores = sorted(zip(classification_results['labels'], classification_results['scores']), key=lambda x: x[1], reverse=True)

    # Heuristic: Select segments above a certain threshold (e.g., `temperature` confidence)
    for label, score in sorted_labels_scores:
        if score > temperature:
            classified_segments_with_scores.append(f"{label} ({score:.2f})") # Format label and score

    segments_message = ""
    if classified_segments_with_scores:
        segments_message = "Esta información es válida para los segmentos: " + ", ".join(classified_segments_with_scores) + "."
    else:
        segments_message = (
            "No se pudieron clasificar segmentos específicos para esta pregunta con alta confianza (umbral: "
            f"{temperature:.2f}). Por favor, intente reformular su pregunta o asegúrese de que se alinee con "
            "los segmentos definidos."
        )

    # Format the response for executive output, including segment classification
    executive_response = f"**Respuesta del Agente:**\n\n{response_text}\n\n{segments_message}\n\n---"

    # Update chat history for the Gradio Chatbot component
    chat_history.append({'role': 'user', 'content': user_question})
    chat_history.append({'role': 'assistant', 'content': response_text})

    progress(1, desc="Finalizado.")
    return chat_history, executive_response, "" # Return updated history, formatted response, and clear user input

def clear_all():
    return [], "", userdata.get("GEMINI_API_KEY"), 0.2, "gemini-2.5-flash", default_pages_data_for_df_for_ui, ""

# Gradio Interface
with gr.Blocks(title="Chatbot comercial segmentado", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📈 Chatbot comercial segmentado")
    gr.Markdown("Una app sencilla para atención segmentada")

    with gr.Sidebar():
        gr.Markdown("### ⚙️ Configuración")
        gemini_api_key_input = gr.Textbox(label="API Key de Gemini", type="password", info="Introduce tu API Key de Google Gemini.", value=userdata.get("GEMINI_API_KEY"))
        temperature_slider = gr.Slider(minimum=0, maximum=1, step=0.1, value=0.2, label="Temperatura", info="Controla la creatividad del modelo (0=más predecible, 1=más creativo).")
        gr.Markdown("[Obtener API Key](https://aistudio.google.com/api-keys)")

        # Modified pages_dataframe to only include URLs
        pages_dataframe = gr.Dataframe(
            headers=["Link html a descargar"],
            datatype=["str"],
            value=default_pages_data_for_df_for_ui,
            label="Definir páginas a descargar",
            interactive=True,
            type="array",
            col_count=(1, "fixed") # Set col_count to 1 to disable adding/deleting columns
        )
        clear_button = gr.Button("Limpiar Conversación y Reiniciar 🗑️")

    # Main Chatbot interface
    model_selector = gr.Radio(
        ["gemini-2.5-flash", "gemini-3.0-flash"], # Gemini 3.0 Flash is not yet public, but kept as per request for future-proofing
        label="Seleccionar Modelo",
        value="gemini-2.5-flash",
        info="Elige el modelo Gemini para generar respuestas."
    )
    chatbot = gr.Chatbot(label="Chat", type='messages', allow_tags=False)

    # Group message input and submit button in a row for better responsiveness
    with gr.Row():
        msg = gr.Textbox(label="Tu pregunta", scale=4)
        submit_btn = gr.Button("Enviar 🚀", scale=1)

    response_output = gr.Textbox(label="Respuesta del Agente (Formato Ejecutivo)", interactive=False, lines=7)

    # Connect components
    submit_btn.click(
        rag_chat,
        inputs=[
            gemini_api_key_input,
            temperature_slider,
            model_selector,
            pages_dataframe,
            chatbot,
            msg
        ],
        outputs=[chatbot, response_output, msg]
    )
    msg.submit(
        rag_chat,
        inputs=[
            gemini_api_key_input,
            temperature_slider,
            model_selector,
            pages_dataframe,
            chatbot,
            msg
        ],
        outputs=[chatbot, response_output, msg]
    )

    clear_button.click(
        clear_all,
        inputs=[],
        outputs=[
            chatbot,
            response_output,
            gemini_api_key_input,
            temperature_slider,
            model_selector,
            pages_dataframe,
            msg
        ]
    )

demo.launch(debug=True, share=False)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

/tmp/ipykernel_7927/3993978221.py:243: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Chatbot comercial segmentado", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_7927/3993978221.py:272: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat", type='messages', allow_tags=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
